In [91]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt 
import seaborn as sns

In [92]:
# The dataset created by `prepare_dataset.py`
df = pd.read_csv('training_dataset.csv')
df.head()


,id,route,flight_class,days_to_departure,total_seats,booked_seats,remaining_seats,current_price,base_fare,booking_date,...,is_weekend,is_holiday_window,petrol_price,diesel_price,usd_to_pkr,competitor_min_price,competitor_avg_price,price_vs_competitor_ratio,competitor_data_is_real,demand_ratio
0,30001,KHI-PEW,Economy,24,180,175,5,8357.95,14859.005505,2026-07-01,...,1,0,340.273076,376.121529,273.391871,NaN,NaN,NaN,False,0.972222
1,30002,KHI-LHE,Economy,43,180,26,154,19266.08,14734.237926,2026-07-01,...,0,1,329.595878,377.099243,282.725161,7485.0,8569.0,2.248346,True,0.144444
2,30003,KHI-DXB,Business,12,180,155,25,78407.86,110311.068521,2026-07-01,...,0,0,331.244788,390.474898,287.402815,NaN,NaN,NaN,False,0.861111
3,30004,KHI-PEW,Business,18,180,103,77,40774.73,42935.468414,2026-07-01,...,1,0,340.349826,382.774175,276.779991,NaN,NaN,NaN,False,0.572222
4,30005,KHI-LHE,Business,9,180,159,21,30251.57,42742.049594,2026-07-01,...,0,0,328.300406,392.101842,285.472981,7485.0,8569.0,3.530350,True,0.883333


In [63]:
df.columns

Index(['id', 'route', 'flight_class', 'days_to_departure', 'total_seats',
       'booked_seats', 'remaining_seats', 'current_price', 'base_fare',
       'booking_date', 'time_of_day', 'day_of_week', 'is_weekend',
       'is_holiday_window', 'petrol_price', 'diesel_price', 'usd_to_pkr',
       'competitor_min_price', 'competitor_avg_price',
       'price_vs_competitor_ratio', 'competitor_data_is_real', 'demand_ratio'],
      dtype='str')

In [64]:
print("Shape :", df.shape)               # rows × columns
print("\nColumns and dtypes:")
print(df.dtypes)


Shape : (15000, 22)

Columns and dtypes:
id                             int64
route                            str
flight_class                     str
days_to_departure              int64
total_seats                    int64
booked_seats                   int64
remaining_seats                int64
current_price                float64
base_fare                    float64
booking_date                     str
time_of_day                    int64
day_of_week                    int64
is_weekend                     int64
is_holiday_window              int64
petrol_price                 float64
diesel_price                 float64
usd_to_pkr                   float64
competitor_min_price         float64
competitor_avg_price         float64
price_vs_competitor_ratio    float64
competitor_data_is_real         bool
demand_ratio                 float64
dtype: object


In [ ]:
# Retrain the demand model from scratch

df = pd.read_csv('training_dataset.csv')

COLS_TO_DROP = [
    'id',
    'total_seats'
    'booking_date',
    'remaining_seats'
]

X = df.drop(columns=COLS_TO_DROP + ['demand_ratio'])
y = df['demand_ratio']

# Fill missing competitor-related values
X['competitor_min_price'] = X['competitor_min_price'].fillna(0)
X['competitor_avg_price'] = X['competitor_avg_price'].fillna(0)
X['price_vs_competitor_ratio'] = X['price_vs_competitor_ratio'].fillna(0)
X['competitor_data_is_real'] = X['competitor_data_is_real'].astype(int)

# One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=['route', 'flight_class', 'time_of_day'],
    drop_first=False
)

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train XGBoost regressor
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    monotone_constraints={'current_price': -1, 'price_vs_competitor_ratio': -1}
    # -1 = jaise ye feature barhta hai, prediction (demand) hamesha kam honi chahiye
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_val)
rmse = mean_squared_error(y_val, y_pred, squared=False)
r2 = r2_score(y_val, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Save trained model
joblib.dump(model, 'demand_model.pkl')
print("Model saved as demand_model.pkl")

# Feature importance
xgb.plot_importance(model, max_num_features=15, importance_type='gain')
plt.tight_layout()
plt.show()
